#  Arm Assembly Primer : Functions and labels
# 1. Our first assembly code
* Let's compile this tiny C function into assembly code


In [ ]:
%%writefile add1.c
int add1(int x) {
  return x + 1;
}

* `-S` tells the compiler to produce assembly code (`add1.s`) instead of an executable


In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang -O1 -S add1.c

In [ ]:
%%bash
cat add1.s

* You will see something like this

```
	.file	"add1.c"
	.text
	.globl	add1                            // -- Begin function add1
	.p2align	2
	.type	add1,@function
add1:                                   // @add1
	.cfi_startproc
// %bb.0:
	add	w0, w0, #1
	ret
.Lfunc_end0:
	.size	add1, .Lfunc_end0-add1
	.cfi_endproc
                                        // -- End function
	.ident	"clang version ..."
	.section	".note.GNU-stack","",@progbits
	.addrsig
```

* There are four kinds of lines
  * a name followed by `:` at the beginning of a line (`add1:`) is a <font color="blue">label</font>, a name given to the location of the instruction that follows it
  * an indented word followed by operands (`add w0, w0, #1`, `ret`) is an <font color="blue">instruction</font>
  * an indented line starting with `.` (`.text`, `.globl add1`, `.cfi_startproc`, ...) is a <font color="blue">directive</font>, extra information for the assembler and the linker (e.g., `.globl add1` says other files may refer to `add1`). You can ignore them for now
  * anything after `//` is a <font color="blue">comment</font>
* So the real content is just this

```
add1:
	add	w0, w0, #1
	ret
```

* `add w0, w0, #1` computes `w0 + 1` and puts the result into `w0`; `w0` is a _register_ (a variable inside the CPU; more on this later) holding the parameter `x`, and `#1` is the constant 1
* `ret` returns to the caller
* <font color="blue">A function is translated into a label (the function name) followed by a sequence of instructions.</font> Calling the function means jumping to the label

# 2. Hiding directives
* Directives clutter the output. From now on we hide them with `grep`; `grep -vE '^\s+\.|^\s*//|^\.Lfunc'` shows lines _except_ (`-v`) those matching any of
  * `^\s+\.` : starting with spaces/tabs followed by `.` (directives)
  * `^\s*//` : starting with `//` (comment-only lines)
  * `^\.Lfunc` : starting with `.Lfunc` (labels marking the end of functions, not used by instructions)


In [ ]:
%%bash
grep -vE '^\s+\.|^\s*//|^\.Lfunc' add1.s

* You can also visualize the assembly code. With only one instruction sequence, it is a single box now, but it will become more interesting when we see `if` and loops


In [ ]:
import vis_asm
vis_asm.vis_asm_file("add1.s")

# 3. Function names in C++
* Let's compile exactly the same function as C++ (the file name ends with `.cc`, and the compiler is `clang++`)


In [ ]:
%%writefile add1_cc.cc
int add1(int x) {
  return x + 1;
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -O1 -S add1_cc.cc
grep -vE '^\s+\.|^\s*//|^\.Lfunc' add1_cc.s

* The instructions are the same, but the label is now `_Z4add1i`, not `add1`
* This is because C++ allows different functions to have the same name, as long as their parameter types differ (_overloading_); labels, on the other hand, must be unique
* So a C++ compiler encodes the parameter types into the label. This is called <font color="blue">name mangling</font>


In [ ]:
%%writefile overload.cc
int add1(int x) {
  return x + 1;
}

double add1(double x) {
  return x + 1;
}

extern "C" int add1_c(int x) {
  return x + 1;
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -O1 -S overload.cc
grep -vE '^\s+\.|^\s*//|^\.Lfunc' overload.s

* `add1(int)` becomes `_Z4add1i` and `add1(double)` becomes `_Z4add1d`
  * `_Z` : "this is a mangled name"
  * `4add1` : the function name, preceded by its length
  * `i`, `d` : the parameter types (`int`, `double`)
  * other common letters: `l` (`long`), `f` (`float`), `c` (`char`), `v` (no parameters), `P` (pointer; `Pl` = `long *`), `K` (`const`)
* The return type is not part of the name
* `extern "C"` turns mangling off, so `add1_c` keeps its name (this is how C++ programs call functions written in C, and vice versa)
* `c++filt` translates a mangled name back into the C++ notation


In [ ]:
%%bash
echo _Z4add1d | c++filt
c++filt < overload.s | grep -vE '^\s+\.|^\s*//|^\.Lfunc'

# 4. Summary
* A function becomes a label followed by instructions
* The label is the function name in C, and a mangled name (`_Z...`) in C++
* When you look for a function in assembly code, search for its name; for C++, `c++filt` helps